# AI City — VLM re-rank top-10 (Qwen2-VL-2B) → đẩy R@1 / mAP

Lấy **top-10 ảnh cuối** từ pipeline X-VLM (`answer.txt`) → cho **Qwen2-VL-2B** chọn ảnh khớp nhất với
caption → đưa lên **rank-1**. Ảnh GT đã rớt khỏi top-10 thì chấp nhận mất (rất ít — R@10 ~99%).

**KERNEL RIÊNG** (không chạy chung X-VLM): Qwen cần `transformers` mới, X-VLM dùng bản cũ → phải tách.

**KHÔNG cần upload answer.txt.** Thay vào đó **Add Input → Notebook Output** của lần chạy X-VLM
(`aicity-inference-v1`) — nó đã chứa `answer.txt`. Notebook này tự dò.

**Add Input (chỉ 4 thứ — KHÔNG cần dbsn/inference/training ở đây):**
1. `xvlm-gallery` (ảnh) · 2. `xvlm-queries` (query_text.json + query_index.txt) · 3. `xvlm-gt` (ground_truth.txt) ·
4. **Output của notebook X-VLM** (chứa `answer.txt`) — qua *Add Input → Notebook Output*, KHÔNG phải dataset.
5. **Qwen2-VL-2B-Instruct** — add qua **Kaggle Models** (search "Qwen2-VL-2B") để **KHÔNG phụ thuộc HuggingFace**.

**Phần cứng:** Qwen2-VL-**2B fp16 ~5GB → 1×T4 dư sức, KHÔNG cần quantize** (giữ full độ chính xác).
**Thời gian:** ~1-2s/query × ~1978 ≈ **~0.5-1h**. **Có RESUME** — bị dừng thì chạy lại tiếp.

In [ ]:
# [1/6] GPU + CONFIG
import glob, os
from pathlib import Path
import torch
ng = torch.cuda.device_count()
print("GPUs:", ng, [torch.cuda.get_device_name(i) for i in range(ng)])

# ---------------- CONFIG ----------------
HF_FALLBACK_ID = "Qwen/Qwen2-VL-2B-Instruct"   # 2B: 1xT4 fp16 thoai mai. (doi "Qwen/Qwen2.5-VL-3B-Instruct" neu muon)
QUANT      = "none"          # 2B -> "none" (fp16 day du). Chi dat "8bit" neu dung model 7B tren 1 GPU.
TOPK       = 10              # so ung vien dua vao VLM
MAX_PIXELS = 512 * 28 * 28   # tran vision-token/anh (giam -> nhanh + it VRAM); KHONG ha qua thap
MIN_PIXELS = 256 * 28 * 28
SHUFFLE    = True            # xao thu tu 10 anh de khu positional-bias
WORK = Path("/kaggle/working"); WORK.mkdir(exist_ok=True)
PICKS = WORK / "vlm_picks.json"        # file resume

def find(pat):
    return sorted(glob.glob(f"/kaggle/input/**/{pat}", recursive=True))

# tu tim thu muc model Qwen trong input (uu tien -> khoi tai HF)
qcfg = [p for p in find("config.json") if "qwen" in p.lower()]
MODEL_DIR = str(Path(qcfg[0]).parent) if qcfg else HF_FALLBACK_ID
print("MODEL:", MODEL_DIR, "(local, khoi dung HF)" if qcfg else "(se tai tu HuggingFace)")

In [ ]:
# [2/6] Cai transformers MOI + qwen-vl-utils (kernel rieng nen an toan)
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
!pip -q install -U "transformers>=4.49.0" "accelerate>=0.34" qwen-vl-utils hf_transfer
!pip -q install bitsandbytes        # chi dung khi QUANT="8bit"
import transformers
print("transformers", transformers.__version__)

In [ ]:
# [3/6] Nap inputs: answer.txt (top-10 tu Notebook Output X-VLM), captions, GT, map image_id -> file
import json
from pathlib import Path

def one(pat):
    h = find(pat); return h[0] if h else None

ANSWER = one("answer.txt"); QJSON = one("query_text.json"); QIDX = one("query_index.txt"); GTF = one("ground_truth.txt")
assert ANSWER, ("Khong thay answer.txt — Add Input -> Notebook Output cua lan chay X-VLM "
                "(aicity-inference-v1), no chua /outputs/answer.txt")
assert QJSON and QIDX and GTF, f"Thieu file: QJSON={QJSON} QIDX={QIDX} GT={GTF}"
print("answer.txt:", ANSWER)

# map stem -> path anh
exts = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
imgs = [p for p in find("*") if p.lower().endswith(exts)]
stem2path = {Path(p).stem: p for p in imgs}
print(f"anh tim thay: {len(stem2path)}")

raw = open(QJSON, encoding="utf-8").read().strip()
recs = json.loads(raw) if raw.startswith("[") else [json.loads(l) for l in raw.splitlines() if l.strip()]
cap_of = {str(r["query_index"]): r["caption"] for r in recs}

qorder = [l.strip() for l in open(QIDX, encoding="utf-8").read().strip().splitlines()]
gts    = [l.strip() for l in open(GTF,  encoding="utf-8").read().strip().splitlines()]
ans    = [l.split() for l in open(ANSWER, encoding="utf-8").read().strip().splitlines()]
assert len(qorder) == len(gts) == len(ans), f"lech dong: qidx={len(qorder)} gt={len(gts)} answer={len(ans)}"

queries = []
for i, full in enumerate(ans):
    cands = full[:TOPK]
    queries.append(dict(full=full, candidates=cands,
                        paths=[stem2path[c] for c in cands if c in stem2path],
                        caption=cap_of.get(qorder[i], ""), gt=gts[i]))
miss = sum(len(q["paths"]) != len(q["candidates"]) for q in queries)
print(f"queries={len(queries)} | TOPK={TOPK} | query thieu file anh={miss}")
print("vi du caption:", queries[0]["caption"][:90])

In [ ]:
# [4/6] Nap Qwen2-VL-2B (fp16, 1 GPU). Tu chon class cho 2B / 2.5.
import torch
from transformers import AutoProcessor
try:
    from transformers import AutoModelForImageTextToText as VLMClass
except ImportError:
    from transformers import Qwen2VLForConditionalGeneration as VLMClass

kw = dict(device_map="auto", torch_dtype=torch.float16)   # T4 = fp16 (KHONG bf16)
if QUANT == "8bit":
    from transformers import BitsAndBytesConfig
    kw = dict(device_map="auto", quantization_config=BitsAndBytesConfig(load_in_8bit=True))
model = VLMClass.from_pretrained(MODEL_DIR, **kw).eval()
processor = AutoProcessor.from_pretrained(MODEL_DIR, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
print("VLM san sang | QUANT =", QUANT)
print("VRAM/GPU:", [f"{torch.cuda.memory_allocated(i)/2**30:.1f}G" for i in range(torch.cuda.device_count())])

In [ ]:
# [5/6] VONG RE-RANK (1 query/lan, resume, try/except chong crash)
import json, random, re, time, torch
from qwen_vl_utils import process_vision_info

picks = json.load(open(PICKS)) if PICKS.exists() else {}     # resume: {query_idx: image_id chon}
random.seed(0)

def make_prompt(n, cap):
    return ("Below are " + str(n) + " candidate images of a person, numbered 1 to " + str(n) +
            ", followed by a text description. Pick the ONE image that best matches the description. "
            "Reply with ONLY the number (1-" + str(n) + "), nothing else.\nDescription: " + str(cap))

t0 = time.time(); done0 = len(picks)
for i, q in enumerate(queries):
    if str(i) in picks:
        continue
    paths, cands = q["paths"], q["candidates"]
    if not paths:
        picks[str(i)] = cands[0] if cands else ""
        continue
    order = list(range(len(paths)))
    if SHUFFLE:
        random.shuffle(order)
    content = [{"type": "image", "image": paths[o]} for o in order]
    content.append({"type": "text", "text": make_prompt(len(paths), q["caption"])})
    msgs = [{"role": "user", "content": content}]
    try:
        text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        ii, vi = process_vision_info(msgs)
        inp = processor(text=[text], images=ii, videos=vi, padding=True, return_tensors="pt").to(model.device)
        with torch.no_grad():
            gen = model.generate(**inp, max_new_tokens=8, do_sample=False)
        out = processor.batch_decode(gen[:, inp.input_ids.shape[1]:], skip_special_tokens=True)[0]
        m = re.search(r"\d+", out)
        sel = int(m.group()) - 1 if m else 0
        sel = sel if 0 <= sel < len(paths) else 0
        picks[str(i)] = cands[order[sel]]           # map vi tri (da xao) -> image_id that
    except Exception as e:
        picks[str(i)] = cands[0]
        if i % 200 == 0:
            print("warn q", i, repr(e)[:120])
    if i % 50 == 0:
        json.dump(picks, open(PICKS, "w"))
        torch.cuda.empty_cache()
        el = (time.time() - t0) / 60
        dn = len(picks) - done0
        eta = el / max(dn, 1) * (len(queries) - len(picks))
        print(f"{len(picks)}/{len(queries)}  {el:.1f}m  ETA ~{eta:.0f}m")
json.dump(picks, open(PICKS, "w"))
print("XONG re-rank:", len(picks), "queries")

In [ ]:
# [6/6] METRICS truoc/sau VLM + ghi answer_vlm.txt
import json
import numpy as np
picks = json.load(open(PICKS))

def metrics(get_rank):
    ranks = [get_rank(i, q) for i, q in enumerate(queries)]
    rr = np.array([1.0 / r if r else 0.0 for r in ranks])
    valid = [r for r in ranks if r]
    return dict(mAP=float(rr.mean()),
                R1=float(np.mean([r == 1 for r in valid])) if valid else 0.0,
                R5=float(np.mean([r <= 5 for r in valid])) if valid else 0.0,
                R10=float(np.mean([r <= 10 for r in valid])) if valid else 0.0)

def before(i, q):
    return q["full"].index(q["gt"]) + 1 if q["gt"] in q["full"] else None

def after(i, q):
    p = picks.get(str(i))
    new = ([p] + [x for x in q["full"] if x != p]) if p in q["full"] else q["full"]
    return new.index(q["gt"]) + 1 if q["gt"] in new else None

B, A = metrics(before), metrics(after)
print(f"{'':8s}{'mAP':>8}{'R@1':>8}{'R@5':>8}{'R@10':>8}")
print(f"{'X-VLM':8s}{B['mAP']:8.4f}{B['R1']:8.4f}{B['R5']:8.4f}{B['R10']:8.4f}")
print(f"{'+VLM':8s}{A['mAP']:8.4f}{A['R1']:8.4f}{A['R5']:8.4f}{A['R10']:8.4f}")
print(f"{'delta':8s}{A['mAP']-B['mAP']:+8.4f}{A['R1']-B['R1']:+8.4f}{A['R5']-B['R5']:+8.4f}{A['R10']-B['R10']:+8.4f}")

with open("/kaggle/working/answer_vlm.txt", "w") as f:
    for i, q in enumerate(queries):
        p = picks.get(str(i))
        new = ([p] + [x for x in q["full"] if x != p]) if p in q["full"] else q["full"]
        f.write(" ".join(new) + "\n")
print("\nLuu: /kaggle/working/answer_vlm.txt + vlm_picks.json")

## Cách chạy (đã bỏ phần upload answer.txt)

1. **Settings** → Accelerator = **GPU T4** (1 cái đủ cho 2B) · Internet **ON**.
2. **Add Input**:
   - `xvlm-gallery`, `xvlm-queries`, `xvlm-gt` (3 dataset bạn đã có).
   - **Notebook Output** của lần chạy X-VLM (`aicity-inference-v1`) → có sẵn `answer.txt`. *(Add Input → tab "Notebook Output" / "Your Work", KHÔNG tạo dataset mới.)*
   - **Qwen2-VL-2B-Instruct** qua **Kaggle Models** (search "Qwen2-VL-2B").
   - *(KHÔNG cần add dbsn-bank / inference / training ở notebook này.)*
3. **Import** notebook này → **Run All**.
4. **Bị dừng?** Run All lại — cell [5] đọc `vlm_picks.json`, chạy tiếp từ chỗ dở.

**Đọc kết quả (cell [6]):** so `X-VLM` vs `+VLM`. Kỳ vọng R@1 nhảy mạnh (regime R@10≈99% → VLM đẩy ảnh đúng lên 1), mAP tăng theo.

**Không phụ thuộc HF:** add Qwen qua Kaggle Models → cell [1] tự dò `MODEL_DIR` local → 0 request HF. **Không quantize** (2B fp16 đầy đủ). Nếu sau muốn 7B: đổi `HF_FALLBACK_ID` + bật T4×2 hoặc `QUANT="8bit"`.

⚠️ Số này trên no-distractor + caption chi tiết → cao hơn đề thi thật (distractor + caption ngắn).